# Downloads parquet data with `pudl` project

In [1]:
import duckdb
from pathlib import Path
from dotenv import load_dotenv
import os
import certifi
import pandas as pd

PUDL distributes the EPA CEMS data as Parquet files on AWS S3 via the AWS Open Data Registry. The smart way to access it isn't to download the multi-GB hourly file — it's to query S3 directly with DuckDB, which fetches only the row groups you need thanks to Parquet's predicate pushdown

## Connect and Install the S3 Extension

In [2]:
# 1. Connect to an in-memory database
con = duckdb.connect()

In [3]:
# 2. Install and load the httpfs extension for S3 support
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"SET ca_cert_file='{certifi.where()}';")

In [4]:
# PUDL's bucket is public — no AWS credentials needed
con.execute("SET s3_region='us-west-2';")

In [5]:
# Disable SSL verification for public S3 buckets
con.execute("SET s3_use_ssl=true;")
con.execute("SET enable_external_file_cache=false;")

## Discover the Schema

In [6]:
load_dotenv()
pudl_cems_url = os.environ.get("PUDL_CEMS_URL")
print(f"Downloading CEMS data from {pudl_cems_url}...")

In [7]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{pudl_cems_url}')").df()
print(schema)

                  column_name column_type null   key default extra
0                       state     VARCHAR  YES  None    None  None
1                plant_id_epa      BIGINT  YES  None    None  None
2       emissions_unit_id_epa     VARCHAR  YES  None    None  None
3        operating_time_hours      DOUBLE  YES  None    None  None
4               gross_load_mw      DOUBLE  YES  None    None  None
5                so2_mass_lbs      DOUBLE  YES  None    None  None
6   so2_mass_measurement_code     VARCHAR  YES  None    None  None
7                nox_mass_lbs      DOUBLE  YES  None    None  None
8   nox_mass_measurement_code     VARCHAR  YES  None    None  None
9               co2_mass_tons      DOUBLE  YES  None    None  None
10  co2_mass_measurement_code     VARCHAR  YES  None    None  None
11         heat_content_mmbtu      DOUBLE  YES  None    None  None
12                       year      BIGINT  YES  None    None  None
13               plant_id_eia      BIGINT  YES  None    None  

## Test Query

Before pulling real data, run something cheap to confirm everything works end-to-end. The test below tries to return the number of lines of hourly records of one day in the period specified in the query

In [8]:
test = con.execute(f"""
    SELECT COUNT(*) AS n_rows
    FROM read_parquet('{pudl_cems_url}')
    WHERE operating_datetime_utc >= '2021-09-01'
    AND operating_datetime_utc <  '2021-09-02'
""").df()
print(test)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_rows
0   98760


## Aggregate Hourly to Daily Across All Four Study Periods

In [14]:
# Paper's four study periods
PERIODS = [
    ("2021-04-01", "2021-05-20"),  # train/val (note: end-exclusive)
    ("2021-09-01", "2021-10-01"),  # test 1
    ("2022-04-01", "2022-04-30"),  # test 2
    ("2022-09-01", "2022-09-29"),  # test 3
]
# Build a SQL OR condition for the four periods
PERIOD_CLAUSES = " OR ".join(
    f"(operating_datetime_utc >= '{start}' AND operating_datetime_utc < '{end}')"
    for start, end in PERIODS
)
QUERY = f"""
SELECT
    plant_id_epa,
    state,
    CAST(operating_datetime_utc AS DATE)        AS date,
    SUM(co2_mass_tons)   * 0.90718474           AS co2_metric_tons,
    SUM(gross_load_mw)                          AS gross_load_mwh,
    SUM(operating_time_hours)                   AS total_operating_hours,
    SUM(heat_content_mmbtu)                     AS total_heat_mmbtu,
    COUNT(*)                                    AS n_hours_reported
FROM read_parquet('{pudl_cems_url}')
WHERE ({PERIOD_CLAUSES})
GROUP BY plant_id_epa, CAST(operating_datetime_utc AS DATE), state
ORDER BY plant_id_epa, date
"""
df = con.execute(QUERY).df()
df["date"] = pd.to_datetime(df["date"])
# print(f"Unfiltered: {len(df):,} rows, {df['plant_id_epa'].nunique():,} plants")
# print(f"Rows: {len(df):,}")
# print(f"Unique plants: {df['plant_id_epa'].nunique():,}")
# print(f"Date range: {df['date'].min()} to {df['date'].max()}")
# df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [15]:
PAPER_PLANTS = [583, 590, 513, 592]
PAPER_SAMPLES = [20306, 14464, 11213, 15243]


def report(df_in, label):
    print(f"\n=== {label} ===")
    print(f"Total: {len(df_in):>7,} rows, {df_in['plant_id_epa'].nunique():>5,} plants")
    print(
        f"{'Period':30s} {'Plants':>8s} {'(paper)':>8s} {'Δ%':>6s} {'Rows':>7s} {'(paper)':>8s}"
    )
    for (start, end), pp, ps in zip(PERIODS, PAPER_PLANTS, PAPER_SAMPLES):
        sub = df_in[(df_in["date"] >= start) & (df_in["date"] < end)]
        n_p = sub["plant_id_epa"].nunique()
        n_r = len(sub)
        delta = (n_p - pp) / pp * 100
        print(
            f"  {start} to {end[:7]:8s}  {n_p:8d} {pp:8d} {delta:+5.1f}% {n_r:7d} {ps:8d}"
        )

In [16]:
df.head()

,plant_id_epa,state,date,co2_metric_tons,gross_load_mwh,total_operating_hours,total_heat_mmbtu,n_hours_reported
0,3,AL,2021-04-01,17944.749187,30737.0,96.0,248698.8,192
1,3,AL,2021-04-02,17937.945301,31062.0,96.0,250361.7,192
2,3,AL,2021-04-03,19261.164963,32953.0,96.0,266051.5,192
3,3,AL,2021-04-04,17990.471297,31021.0,96.0,250107.7,192
4,3,AL,2021-04-05,18994.815523,32192.0,96.0,260309.5,192


In [20]:
report(df, "Step 0 — Raw (CONUS not yet applied)")

# Filter A: CONUS only — the paper explicitly says this
NON_CONUS = {"AK", "HI", "PR", "VI", "GU", "MP", "AS"}
dfA = df[~df["state"].isin(NON_CONUS)].copy()
report(dfA, "Step A — CONUS only")


=== Step 0 — Raw (CONUS not yet applied) ===
Total: 185,331 rows, 1,390 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05       1376      583 +136.0%   67088    20306
  2021-09-01 to 2021-10       1363      590 +131.0%   40890    14464
  2022-04-01 to 2022-04       1361      513 +165.3%   39413    11213
  2022-09-01 to 2022-09       1355      592 +128.9%   37940    15243

=== Step A — CONUS only ===
Total: 185,059 rows, 1,388 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05       1374      583 +135.7%   66990    20306
  2021-09-01 to 2021-10       1361      590 +130.7%   40830    14464
  2022-04-01 to 2022-04       1359      513 +164.9%   39355    11213
  2022-09-01 to 2022-09       1353      592 +128.5%   37884    15243


In [21]:
# Filter B: positive CO2 and positive generation
dfB = dfA[(dfA["co2_metric_tons"] > 0) & (dfA["gross_load_mwh"] > 0)].copy()
report(dfB, "Step B — + Positive CO2 and load")


=== Step B — + Positive CO2 and load ===
Total:  94,832 rows, 1,136 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05       1045      583 +79.2%   30976    20306
  2021-09-01 to 2021-10       1075      590 +82.2%   22895    14464
  2022-04-01 to 2022-04        994      513 +93.8%   18478    11213
  2022-09-01 to 2022-09       1056      592 +78.4%   22483    15243


In [22]:
# Filter C: meaningful operation (not a trickle)
dfC = dfB[dfB["total_operating_hours"] >= 1].copy()
report(dfC, "Step C — + At least 1 operating-hour summed across units")


=== Step C — + At least 1 operating-hour summed across units ===
Total:  93,715 rows, 1,131 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05       1039      583 +78.2%   30579    20306
  2021-09-01 to 2021-10       1070      590 +81.4%   22621    14464
  2022-04-01 to 2022-04        987      513 +92.4%   18224    11213
  2022-09-01 to 2022-09       1050      592 +77.4%   22291    15243


In [23]:
# Filter D: near-complete daily coverage
dfD = dfC[dfC["n_hours_reported"] >= 20].copy()
report(dfD, "Step D — + At least 20 hours of records that day")


=== Step D — + At least 20 hours of records that day ===
Total:  93,715 rows, 1,131 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05       1039      583 +78.2%   30579    20306
  2021-09-01 to 2021-10       1070      590 +81.4%   22621    14464
  2022-04-01 to 2022-04        987      513 +92.4%   18224    11213
  2022-09-01 to 2022-09       1050      592 +77.4%   22291    15243


In [24]:
strict_query = f"""
SELECT
    plant_id_epa,
    state,
    CAST(operating_datetime_utc AS DATE) AS date,
    SUM(co2_mass_tons) * 0.90718474   AS co2_metric_tons,
    SUM(gross_load_mw)                AS gross_load_mwh,
    SUM(operating_time_hours)         AS total_operating_hours,
    COUNT(*)                          AS n_hour_records,
    COUNT(co2_mass_tons)              AS n_co2_non_null,
    COUNT(gross_load_mw)              AS n_load_non_null
FROM read_parquet('{pudl_cems_url}')
WHERE ({PERIOD_CLAUSES})
GROUP BY plant_id_epa, state, CAST(operating_datetime_utc AS DATE)
"""
df_strict = con.execute(strict_query).df()
df_strict["date"] = pd.to_datetime(df_strict["date"])

# Note: COUNT(col) counts non-null values; COUNT(*) counts all rows
# Days where n_co2_non_null < n_hour_records have null CO2 hours

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [25]:
NON_CONUS = {"AK", "HI", "PR", "VI", "GU", "MP", "AS"}
df_strict = df_strict[~df_strict["state"].isin(NON_CONUS)]

In [26]:
dfE1 = df_strict[
    (df_strict["co2_metric_tons"] > 0) & (df_strict["gross_load_mwh"] > 0)
].copy()
report(dfE1, "E1 — Positive CO2 and load")


=== E1 — Positive CO2 and load ===
Total:  94,832 rows, 1,136 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05       1045      583 +79.2%   30976    20306
  2021-09-01 to 2021-10       1075      590 +82.2%   22895    14464
  2022-04-01 to 2022-04        994      513 +93.8%   18478    11213
  2022-09-01 to 2022-09       1056      592 +78.4%   22483    15243


In [27]:
dfE2 = dfE1[dfE1["n_co2_non_null"] == dfE1["n_hour_records"]].copy()
report(dfE2, "E2 — + No null CO2 hours within reported records")


=== E2 — + No null CO2 hours within reported records ===
Total:  29,645 rows,   601 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05        431      583 -26.1%    8434    20306
  2021-09-01 to 2021-10        477      590 -19.2%    7809    14464
  2022-04-01 to 2022-04        368      513 -28.3%    4963    11213
  2022-09-01 to 2022-09        493      592 -16.7%    8439    15243


In [30]:
dfE3 = dfE1[dfE1["n_load_non_null"] == dfE1["n_hour_records"]].copy()
report(dfE3, "E3 — + No null gross load hours either")


=== E3 — + No null gross load hours either ===
Total:  29,440 rows,   598 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05        429      583 -26.4%    8345    20306
  2021-09-01 to 2021-10        475      590 -19.5%    7753    14464
  2022-04-01 to 2022-04        368      513 -28.3%    4963    11213
  2022-09-01 to 2022-09        490      592 -17.2%    8379    15243


In [31]:
dfE4 = dfE1[dfE1["n_hour_records"] >= 24].copy()
report(dfE4, "E4 — + Full 24-hour reporting")


=== E4 — + Full 24-hour reporting ===
Total:  94,832 rows, 1,136 plants
Period                           Plants  (paper)     Δ%    Rows  (paper)
  2021-04-01 to 2021-05       1045      583 +79.2%   30976    20306
  2021-09-01 to 2021-10       1075      590 +82.2%   22895    14464
  2022-04-01 to 2022-04        994      513 +93.8%   18478    11213
  2022-09-01 to 2022-09       1056      592 +78.4%   22483    15243


## Save and Verify

In [51]:
out_path = Path("data/processed/epa_daily.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(out_path, index=False, compression="snappy")
print(f"Saved {len(df):,} rows to {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

Saved 94,831 rows to data/processed/epa_daily.parquet
File size: 1.3 MB


## Sanity Checks Against the Paper

In [52]:
import pandas as pd

df["date"] = pd.to_datetime(df["date"])

period_labels = ["2021-04 train", "2021-09 test", "2022-04 test", "2022-09 test"]
for (start, end), label in zip(PERIODS, period_labels):
    mask = (df["date"] >= start) & (df["date"] < end)
    sub = df.loc[mask]
    n_plants = sub["plant_id_epa"].nunique()
    n_rows = len(sub)
    print(f"{label:18s}  plants={n_plants:4d}  rows={n_rows:6d}")

2021-04 train       plants=1045  rows= 30976
2021-09 test        plants=1075  rows= 22895
2022-04 test        plants= 994  rows= 18477
2022-09 test        plants=1056  rows= 22483


In [53]:
# How does the row count per plant distribute?
counts_per_plant = (
    df[(df["date"] >= "2021-04-01") & (df["date"] < "2021-05-20")]
    .groupby("plant_id_epa")
    .size()
    .sort_values()
)

print("Plants by number of reported days in period 1:")
print(counts_per_plant.describe())
print("\nDistribution:")
print(counts_per_plant.value_counts().sort_index().head(20))

Plants by number of reported days in period 1:
count    1045.000000
mean       29.642105
std        16.710063
min         1.000000
25%        14.000000
50%        33.000000
75%        46.000000
max        49.000000
dtype: float64

Distribution:
1     29
2     27
3     21
4     25
5     24
6     17
7     17
8     19
9     18
10    15
11    11
12    18
13    12
14    13
15    22
16    19
17     9
18    19
19     9
20    15
Name: count, dtype: int64


In [57]:
period_mask = (df["date"] >= "2021-04-01") & (df["date"] < "2021-05-20")
sub = df.loc[period_mask]

per_plant = (
    sub.groupby("plant_id_epa")
    .agg(
        days_reported=("date", "nunique"),
        total_co2=("co2_metric_tons", "sum"),
        total_load=("gross_load_mwh", "sum"),
    )
    .sort_values("total_co2")
)

print("Plant size distribution (period 1):")
print(per_plant.describe())

# What fraction of plants account for 90% of emissions?
per_plant_sorted = per_plant.sort_values("total_co2", ascending=False)
cumulative = (
    per_plant_sorted["total_co2"].cumsum() / per_plant_sorted["total_co2"].sum()
)
n_for_95pct = (cumulative <= 0.98).sum() + 1
print(f"\n{n_for_95pct} plants account for 95% of total CO2")
print(f"Total plants in period: {len(per_plant)}")

Plant size distribution (period 1):
       days_reported     total_co2    total_load
count    1045.000000  1.045000e+03  1.045000e+03
mean       29.642105  1.527715e+05  2.519702e+05
std        16.710063  2.584308e+05  3.657707e+05
min         1.000000  1.088622e+00  1.000000e+00
25%        14.000000  5.433048e+03  9.780000e+03
50%        33.000000  4.454716e+04  7.825600e+04
75%        46.000000  1.843541e+05  3.642610e+05
max        49.000000  2.181950e+06  2.868472e+06

593 plants account for 95% of total CO2
Total plants in period: 1045
